# 实时验配界面 —— 4 段增益

板子上另开一条程序在跑：**麦克风 → 核 → 耳机**，实时。
这一页**不参与音频**，它只和那条程序交换两个小文件：

| 文件 | 谁写 | 谁读 |
|---|---|---|
| `/dev/shm/fir_tune.json` | 这一页（4 根滑杆拧出来的值） | `fir_live.py` |
| `/dev/shm/fir_status.json` | `fir_live.py`（增益回显 + 电平） | 这一页 |

**所以：这一页关掉、卡住、刷新，声音都不会断。** 反过来也一样 —— 实时那条没起，
这里只会显示"读不到状态"，不会报错。

## 怎么用

1. 戴好耳机。四段插头要**插到底**；插着却录不到声音时见
   `skill/pitfalls/pynq_audio_playback.md` 坑 5 —— **要拔插到电平表跳起来为止**。
2. 从上往下依次运行本页所有单元（菜单 `Run → Run All Cells`）。
3. 点 **① 启动实时**。
4. 拖滑杆 —— 耳机里的声音应该跟着变。

## 这个界面在演示里做的是什么

助听器验配就是这个动作：**哪一段听不见，就把哪一段抬起来**。
听力正常的人四根滑杆都在 0 dB；把低频压下去、高频抬起来，
就是一张"高频听力损失"的听力图。点几个预设能直接听出区别。


In [ ]:
import json
import math
import os
import signal
import subprocess
import sys
import threading
import time

import ipywidgets as W
from IPython.display import display

HERE = os.getcwd()
if HERE not in sys.path:
    sys.path.insert(0, HERE)
from fir_tune import read_status, write_tune, TUNE_PATH, STATUS_PATH

print("ipywidgets", W.__version__)
print("旋钮文件", TUNE_PATH)
print("状态文件", STATUS_PATH)
print("→ 现在读不到状态：实时那条还没起（很正常，下一步点「启动」）"
      if read_status() is None else "→ 状态文件有了，实时那条在跑或跑过一轮")

# 搬样线程绑在 CPU0，而且是纯空转（SCHED_FIFO 20、永不阻塞）——CPU0 上的
# 普通线程会被饿到**几秒才跑一次**（实测 5 秒里只调度了 4 次）。
# ⚠ os.sched_setaffinity(0, …) 只钉**当前这一个线程**，不是整个进程。
# Jupyter 收滑杆消息的是 ZeroMQ 自己的 I/O 线程，它在本单元之前就建好了；
# 只钉主线程的话，消息照样落在 CPU0 上等几秒（2026-10-18 实测：主线程已在
# CPU1，另有 10 个线程仍在 0,1）。所以把 /proc/self/task 下所有线程一起钉过去。
# 此后新建的线程会从创建者继承 CPU1。
def _pin_pid(pid, cpu=1):
    n = 0
    try:
        tids = os.listdir("/proc/%d/task" % pid)
    except OSError:
        return 0
    for _tid in tids:
        try:
            os.sched_setaffinity(int(_tid), {cpu})
            n += 1
        except OSError:
            pass
    return n

try:
    _n = _pin_pid(os.getpid(), 1)
    # 浏览器 → Jupyter 服务(pid 551) → ZMQ → 内核。服务进程也在 CPU0 上排队的话，
    # 只钉内核没用。它的 pid 就是本内核的父进程（2026-10-18 实测：三个线程全在 0,1）。
    _srv = os.getppid()
    try:
        _cmd = open("/proc/%d/cmdline" % _srv, "rb").read().replace(
            b"\0", b" ").decode("utf-8", "replace")
    except OSError:
        _cmd = ""
    if _srv > 1 and "jupyter" in _cmd.lower():
        _m = _pin_pid(_srv, 1)
        print("→ 本内核 %d 个线程 + Jupyter 服务(pid %d) %d 个线程 → CPU1" % (_n, _srv, _m))
    else:
        print("→ 本内核 %d 个线程 → CPU1（父进程不是 Jupyter，没动它）" % _n)
except OSError as _e:
    print("⚠ 钉 CPU1 失败（%s）—— 滑杆响应/界面刷新可能滞后几秒" % _e)


In [ ]:
# ---------------- 4 根滑杆 ----------------
# 分段点来自核的源码（相减式互补 FIR）：b1 = 0~500，b2 = 500~1k，b3 = 1k~2k，b4 = 2k~24k。
BANDS = ["① 0 – 500 Hz", "② 500 – 1 kHz", "③ 1 – 2 kHz", "④ 2 – 24 kHz"]
GAIN_UI_LIMIT = 24.0        # 界面只给到 ±24 dB（核能到 ±42，演示用不了那么狠）

sld = [W.FloatSlider(value=0.0, min=-GAIN_UI_LIMIT, max=GAIN_UI_LIMIT, step=0.5,
                     description=b, continuous_update=True, readout_format="+.1f",
                     style={"description_width": "110px"},
                     layout=W.Layout(width="560px"))
       for b in BANDS]

PRESETS = {
    "平坦：不动（0 dB）":              (0, 0, 0, 0),
    "高频下降：典型老年性听力图":      (-4, -2, 6, 14),
    "只抬辅音：清辅音都在 2 kHz 以上": (-10, -2, 6, 10),
    "压掉低频轰鸣":                    (-16, -6, 2, 4),
    "极端对照：一拖就听出区别":        (-20, -10, 10, 20),
}
PLACEHOLDER = "（选一个套用）"
drop = W.Dropdown(options=[PLACEHOLDER] + list(PRESETS), value=PLACEHOLDER,
                  description="预设", style={"description_width": "60px"},
                  layout=W.Layout(width="430px"))
src = W.Dropdown(options=[("麦克风", "mic"), ("放 wav（没插麦也能验流水线）", "wav")],
                 value="mic", description="音源",
                 style={"description_width": "60px"})
byp = W.ToggleButton(value=False, description="直通（跳过压缩）",
                     layout=W.Layout(width="200px"))


# ---------------- 拧滑杆 → 写旋钮文件 ----------------
# **不是直接把参数据写进核** —— 这边一个寄存器都不碰。
# 实时那条自己在下一块开头把文件读走，所以这一页卡住也不会让音频卡住。
seq = [0]


def push_tune(*_):
    seq[0] += 1
    write_tune({"seq": seq[0],
                "gain_db": [round(s.value, 2) for s in sld],
                "bypass": 1 if byp.value else 0})


for _s in sld:
    _s.observe(push_tune, names="value")
byp.observe(push_tune, names="value")


def on_preset(change):
    if change["new"] in PRESETS:
        for _s, _v in zip(sld, PRESETS[change["new"]]):
            _s.value = _v              # 每改一根都会自己写一次旋钮文件，多写几次不要紧
        drop.value = PLACEHOLDER       # 拨回提示项，同一个预设才能再点一次


drop.observe(on_preset, names="value")


# ---------------- 死机防护的前置检查 ----------------
# 搬样线程是 SCHED_FIFO 20 且永不阻塞，CPU0 上的 ksoftirqd / kworker 若是普通 CFS 类
# 就会被饿死，约 60 秒后 RCU stall → panic。判据用内核里的实际调度策略，
# 不只看服务状态：手工 chrt 过也算数，反过来服务 active 但策略没上去不算。
def hang_fix_state():
    import re

    def _ask(pid):
        q = subprocess.run(["chrt", "-p", pid], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, universal_newlines=True)
        txt = q.stdout
        m = re.search(r"scheduling priority:\s*(\d+)", txt)
        return txt, (int(m.group(1)) if m else None)

    def _fifo(pid):
        txt, prio = _ask(pid)
        return ("SCHED_FIFO" in txt and prio is not None and prio > 20), txt

    def _promote(pid):
        # 现场提权，不靠 30 秒重扫 timer。发现一个、提一个。
        q = subprocess.run(["chrt", "-f", "-p", "50", pid],
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           universal_newlines=True)
        return q.returncode == 0, " ".join(q.stdout.split())[:80]

    notes = []
    promoted = 0
    for cpu in ("0", "1"):
        r = subprocess.run(["pgrep", "-f", "^ksoftirqd/%s$" % cpu],
                           stdout=subprocess.PIPE, stderr=subprocess.DEVNULL,
                           universal_newlines=True)
        pids = r.stdout.split()
        if not pids:
            return False, "找不到 ksoftirqd/%s" % cpu
        ok, txt = _fifo(pids[0])
        if not ok:
            ok2, why = _promote(pids[0])
            if not ok2:
                return False, "ksoftirqd/%s 提权失败：%s" % (cpu, why)
            promoted += 1
        notes.append("ksoftirqd/%s FIFO 50" % cpu)
    # kworker 也是普通 CFS，同样会被搬样线程永久饿死（round12b 的 P21 就是它）。
    # 运行中 workqueue 会按需新建 kworker，在 30 秒重扫之前它们还是 CFS；
    # 以前这里只要撞上一个就报"防护没生效"——板子其实没事，是检查太死。
    # 现在：短命的跳过，没提权的当场提，只有 chrt 自己失败才算真失败。
    r = subprocess.run(["pgrep", "-f", "^kworker/[01]:"],
                       stdout=subprocess.PIPE, stderr=subprocess.DEVNULL,
                       universal_newlines=True)
    n_ok = 0
    for pid in r.stdout.split():
        ok, txt = _fifo(pid)
        if "No such process" in txt:
            continue
        if not ok:
            ok2, why = _promote(pid)
            if not ok2:
                if "No such process" in why:
                    continue
                return False, "kworker pid %s 提权失败：%s" % (pid, why)
            promoted += 1
        n_ok += 1
    notes.append("kworker %d 个 FIFO 50%s" % (
        n_ok, "（现场提权 %d）" % promoted if promoted else ""))
    return True, " / ".join(notes)


# ---------------- 起停实时那条 ----------------
PROC = {"p": None, "log": None}


def start(_):
    p = PROC["p"]
    if p is not None and p.poll() is None:
        print("已经在跑了（pid %d）" % p.pid)
        return
    ok, why = hang_fix_state()
    if not ok:
        print("⚠ 死机防护没生效（%s）。" % why)
        print("  这样启动，实时通路大约一分钟内会把板子挂死。先跑：")
        print("    sudo sh /home/xilinx/install_ksoftirqd_rt.sh")
        print("  然后重启板子（这个服务是开机时生效的）。")
        return
    print("死机防护：%s" % why)
    env = dict(os.environ)
    env.update({"XILINX_XRT": "/usr", "PYTHONUNBUFFERED": "1",
                "LIVE_SRC": src.value, "LIVE_SECONDS": "0",
                "LIVE_GAIN_DB": ",".join("%.2f" % s.value for s in sld),
                "LIVE_SCOPE": "1"})
    if PROC["log"]:
        PROC["log"].close()
    PROC["log"] = open("/dev/shm/fir_live_ui.log", "wb")
    PROC["p"] = subprocess.Popen(
        [sys.executable, os.path.join(HERE, "fir_live.py")], cwd=HERE, env=env,
        stdout=PROC["log"], stderr=subprocess.STDOUT, start_new_session=True)
    print("起来了，pid %d，音源 %s" % (PROC["p"].pid, src.value))
    # 起跑检查。两种最常见的失败都发生在头一两秒：已经有一条在跑了（fir_live.py
    # 自己用文件锁挡住了），或者加载 overlay 失败。两种情况它都会马上退出，
    # 原因写在日志里 —— 直接翻出来，别让人对着一动不动的数字猜。
    time.sleep(2.0)
    rc = PROC["p"].poll()
    if rc is not None:
        print("\n它起跑就退了（退出码 %s）。日志最后几行：" % rc)
        print("  " + open("/dev/shm/fir_live_ui.log", encoding="utf-8",
                          errors="replace").read()[-1200:].strip())
    else:
        print("日志 /dev/shm/fir_live_ui.log")


def stop(_):
    p = PROC["p"]
    if p is None or p.poll() is not None:
        print("没在跑")
        return
    # 发 SIGINT 而不是 SIGTERM：SIGINT 走 KeyboardInterrupt 那条路，
    # 收尾会把音频通路正常关掉（那一下静音）。SIGTERM 是直接打死，耳机里会留一声。
    p.send_signal(signal.SIGINT)
    # 收尾不是"关一下通路"就完了：它要把这次真跑过的 2000 块（20 秒音频）
    # 逐块重算一遍核，做逐位自检。板子上这一步要十几秒，原来只等 8 秒，
    # 于是每次点停止都被强杀：报告截一半、状态文件永远停在做"运行中"。
    print("等它收尾（要重算 2000 块做逐位自检，十几秒）…")
    try:
        p.wait(timeout=60)
        print("停了（pid %d）" % p.pid)
    except subprocess.TimeoutExpired:
        p.kill()
        p.wait()      # 收尸，不然它变成僵尸挂在进程表里
        print("60 秒还没收完，强杀了 —— 报告会缺最后一段")


btn_run = W.Button(description="① 启动实时", button_style="success")
btn_stop = W.Button(description="② 停止", button_style="warning")
btn_run.on_click(start)
btn_stop.on_click(stop)

push_tune()      # 先把当前滑杆位置写进去，这样"先拧好再启动"也生效
display(W.VBox([W.HTML("<b>每段增益</b> —— 拖一下，耳机里应该马上跟着变"),
                *sld, W.HBox([drop, src]), W.HBox([byp, btn_run, btn_stop])]))


In [ ]:
# ---------------- 表针（后台线程，10 Hz） ----------------
# 为什么用后台线程：在单元里写 `while True:` 会把内核占死，滑块全不响应。
meter = W.HTML()
readout = W.HTML()
# 重跑本格时必须把**上一次**那个线程停掉。所以这个字典不能在重跑时换新的：
# 以前每重跑一次就新建一个 Event，旧线程永远收不到 stop，于是一直在后台按
# 10 Hz 往一个已经不在页面上的 widget 里写 —— 白烧 CPU（2026-10-18 看到过
# 两对线程同时在跑）。用 globals 里已有的那一份。
try:
    WATCH
except NameError:
    WATCH = {"stop": threading.Event(), "th": None}


def db_of(rms):
    """int16 域有效值 → dBFS。满量程 32768 = 0 dBFS。"""
    return 20.0 * math.log10(rms / 32768.0) if rms and rms > 0 else -120.0


def lvl_bar(db, width=200):
    pct = max(0.0, min(100.0, (db + 60.0) / 60.0 * 100.0))
    col = "#2ecc71" if db < -12 else ("#f1c40f" if db < -3 else "#e74c3c")
    return ('<span style="display:inline-block;width:%dpx;height:12px;'
            'background:#e8e8e8;border-radius:2px;vertical-align:middle">'
            '<span style="display:inline-block;width:%.0f%%;height:12px;'
            'background:%s;border-radius:2px"></span></span>' % (width, pct, col))


def gain_bar(db, lim=GAIN_UI_LIMIT):
    """中间一条零线：往右是抬、往左是压。"""
    px = max(0.0, min(1.0, abs(db) / lim)) * 55.0
    col = "#2ecc71" if db >= 0 else "#3498db"
    if db >= 0:
        seg = ('<span style="position:absolute;left:55px;width:%.1fpx;height:12px;'
               'background:%s"></span>' % (px, col))
    else:
        seg = ('<span style="position:absolute;left:%.1fpx;width:%.1fpx;height:12px;'
               'background:%s"></span>' % (55.0 - px, px, col))
    return ('<span style="position:relative;display:inline-block;width:110px;'
            'height:12px;background:#e8e8e8;border-radius:2px;vertical-align:middle">'
            '<span style="position:absolute;left:55px;width:1px;height:12px;'
            'background:#999"></span>%s</span>' % seg)


def _watch():
    try:
        os.sched_setaffinity(0, {1})   # 别落在被搬样线程占满的 CPU0 上
    except OSError:
        pass
    while not WATCH["stop"].is_set():
        st = read_status()
        if st is None:
            meter.value = ('<span style="color:#b00">读不到 %s</span> —— '
                           '实时那条还没起？' % STATUS_PATH)
            readout.value = ""
            time.sleep(0.3)
            continue

        app = st.get("gain_applied_db") or [None] * 4
        tgt = st.get("gain_db") or [None] * 4
        rows = []
        for _i, _name in enumerate(BANDS):
            a = 0.0 if app[_i] is None else app[_i]
            t = 0.0 if tgt[_i] is None else tgt[_i]
            rows.append(
                '<tr><td style="padding:2px 10px 2px 0">%s</td>'
                '<td style="padding-right:8px"><b>%+.1f dB</b></td>'
                '<td style="padding-right:8px;color:#888;font-size:12px">'
                '目标 %+.1f</td><td>%s</td></tr>' % (_name, a, t, gain_bar(a)))

        db_in, db_out = db_of(st.get("in_rms")), db_of(st.get("out_rms"))
        act = (db_out - db_in) if (st.get("in_rms") or 0) > 0 else 0.0
        meter.value = (
            '<table style="font-size:13px">%s</table>'
            '<div style="margin-top:8px;font-size:13px">'
            '进核 %s %.1f dBFS&nbsp;&nbsp;&nbsp; 出核 %s %.1f dBFS'
            '&nbsp;&nbsp;&nbsp; 压缩动作 <b>%+.1f dB</b></div>'
            % ("".join(rows), lvl_bar(db_in), db_in, lvl_bar(db_out), db_out, act))

        # 帧有多旧？running=True 但帧很旧 = 那条进程卡住了或者被 kill -9 了。
        # 没有这一条的话，界面会把最后一帧当成"现在"一直显示下去 —— 最坑的假象。
        age = time.time() - (st.get("t") or 0)
        if not st.get("running"):
            tail = ("已停止，自检通过" if st.get("ok")
                    else ("已停止，自检<b style='color:#b00'>没过</b>"
                          if st.get("ok") is False else "已停止"))
        elif age > 1.5:
            tail = ("<b style='color:#b00'>说是运行中，但状态已经 %.1f 秒没更新了 —— "
                    "下面的数字是旧的</b>" % age)
        else:
            tail = "运行中"
        if st.get("error"):
            tail += " <span style='color:#b00'>报错：%s</span>" % st["error"]
        readout.value = (
            '<div style="font-size:13px">第 %s 块&nbsp;&nbsp; 实时 %s'
            '&nbsp;&nbsp; 丢块 %s&nbsp;&nbsp; 搬样 %s ms&nbsp;&nbsp;'
            ' 过核 %s ms&nbsp;&nbsp; %s</div>' % (
                st.get("n", 0),
                ("%.2fx" % st["rt"]) if st.get("rt") else "—",
                st.get("lost_blocks", 0),
                ("%.2f" % st["io_ms"]) if st.get("io_ms") else "—",
                ("%.2f" % st["dsp_ms"]) if st.get("dsp_ms") else "—",
                tail))
        time.sleep(0.1)


# 重跑本单元时先把上一个表针线程停掉 —— 不然两个线程一起往同一个 meter 里写。
WATCH["stop"].set()
if WATCH["th"] is not None:
    WATCH["th"].join(timeout=1.0)
WATCH["stop"].clear()
WATCH["th"] = threading.Thread(target=_watch, daemon=True)
WATCH["th"].start()

display(meter, readout)


In [ ]:
# ---------------- 实时波形（时域 + 频谱） ----------------
# 数据来自 /dev/shm/fir_scope.bin —— fir_live.py **只在加了 LIVE_SCOPE=1 时才写它**。
# 和上面那块表针同一个套路：后台线程读文件、拼一段 HTML（这里是内嵌 SVG），
# 经 ipywidgets 推给浏览器。**音频那条路一个字节都不碰。**
#
# 为什么不用 matplotlib：每帧要把图渲染成 PNG 再 base64 推过去，PYNQ 上
# 一张两面板图要几十毫秒，10 Hz 就吃掉大半个核。SVG 是一小段文字，浏览器自己画。
#
# **纵轴是钉死的、不自动缩放。** 这是关键：自动缩放会把"电平变了"重新归一化掉，
# 看上去永远一样大；钉死满量程才看得出"这一段被压下去了"。
import numpy as np

SCOPE_PATH_UI = "/dev/shm/fir_scope.bin"
SCOPE_N_UI = 960                              # 必须和 fir_live.py 的 LIVE_SCOPE_N 一致
SCOPE_WANT = 16 + 2 * SCOPE_N_UI * 2          # 定长记录：头 16 字节 + 两段 int16

scope = W.HTML()
try:
    SCOPE
except NameError:
    SCOPE = {"stop": threading.Event(), "th": None}

# **波形每秒只画 4 次，而且一根线只取 120 点。**
# 这不是省板子，是省浏览器：每帧要把整段 HTML+SVG 重新塞进页面，浏览器要重新
# 解析、重新排版。10 Hz × 240 点那版实测把浏览器的消息队列压住了 —— 表针
# （另一个 widget）跟着一起晚几秒，看着像"表针慢"，其实是波形把它堵住了。
# 2026-10-18 用 SCOPE["stop"].set() 把波形停掉，表针立刻实时，才定位到这里。
SCOPE_SEC = 0.25
SCOPE_POINTS = 120

W_PX, H_PX = 1000, 170      # 2026-10-18：原来 560×86，太小看不清
FS_UI = 48000.0
DELAY_SAMPLES = 96            # 核的群延迟：出核那一段对应的是 96 个采样之前的进核
SHOW_HZ = 4000.0                              # 四段的三个分段点都在这里面
GRID_HZ = (500.0, 1000.0, 2000.0)


def _poly(xs, ys, color, w=1.0, op=1.0):
    pts = " ".join("%.1f,%.1f" % (x, y) for x, y in zip(xs, ys))
    return ('<polyline points="%s" fill="none" stroke="%s" stroke-width="%.1f" '
            'stroke-opacity="%.2f"/>' % (pts, color, w, op))


def _panel(title, body):
    # viewBox + 100%% 宽：浏览器窗口窄时整体缩放到容器，不会横向截断。
    return ('<div style="font:13px monospace;color:#555;margin:10px 0 3px">%s</div>'
            '<svg viewBox="0 0 %d %d" style="width:100%%;max-width:%dpx;'
            'background:#fafafa;border:1px solid #ddd;display:block">%s</svg>'
            % (title, W_PX, H_PX, W_PX, body))


def _time_panel(a, b):
    # 先补回群延迟：不补的话两条波形横着错开 2 ms，看着像"声音变了"，其实只是延迟。
    a, b = a[:a.size - DELAY_SAMPLES], b[DELAY_SAMPLES:]
    n = a.size
    idx = list(range(0, n, max(1, n // SCOPE_POINTS)))
    half = H_PX / 2.0
    xs = [i / (n - 1) * W_PX for i in idx]

    def ys(v):
        return [half - (float(v[i]) / 32768.0) * (half - 3.0) for i in idx]

    mid = ('<line x1="0" y1="%.1f" x2="%d" y2="%.1f" stroke="#ccc"/>'
           % (half, W_PX, half))
    return mid + _poly(xs, ys(a), "#999", 1.0, 0.7) + _poly(xs, ys(b), "#0a7", 1.3)


def _fft_panel(a, b):
    a, b = a[:a.size - DELAY_SAMPLES], b[DELAY_SAMPLES:]      # 同上，先对齐
    n = a.size
    win = np.hanning(n)
    freqs = np.fft.rfftfreq(n, 1.0 / FS_UI)
    keep = freqs <= SHOW_HZ
    lo, hi = -100.0, 0.0

    def spec(v):
        m = np.abs(np.fft.rfft(v.astype(np.float64) * win)) / (n / 4.0) / 32768.0
        db = np.clip(20.0 * np.log10(np.maximum(m[keep], 1e-9)), lo, hi)
        return [H_PX - (x - lo) / (hi - lo) * H_PX for x in db]

    xs = [f / SHOW_HZ * W_PX for f in freqs[keep]]
    grid = "".join(
        '<line x1="%.1f" y1="0" x2="%.1f" y2="%d" stroke="#e0e0e0" '
        'stroke-dasharray="3,3"/><text x="%.1f" y="10" font-size="9" '
        'fill="#bbb">%gk</text>' % (f / SHOW_HZ * W_PX, f / SHOW_HZ * W_PX,
                                    H_PX, f / SHOW_HZ * W_PX + 2, f / 1000.0)
        for f in GRID_HZ)
    return grid + _poly(xs, spec(a), "#999", 1.0, 0.7) + _poly(xs, spec(b), "#0a7", 1.3)


def _scope_watch():
    try:
        os.sched_setaffinity(0, {1})   # 同上
    except OSError:
        pass
    while not SCOPE["stop"].is_set():
        try:
            raw = open(SCOPE_PATH_UI, "rb").read()
        except OSError:
            scope.value = ('<span style="color:#b00">读不到 %s</span> —— 实时那条'
                           '没在跑，或者启动时没加 <code>LIVE_SCOPE=1</code>。'
                           % SCOPE_PATH_UI)
            SCOPE["stop"].wait(0.5)
            continue
        if len(raw) != SCOPE_WANT:
            scope.value = ('<span style="color:#b00">%s 长度 %d，期望 %d</span>'
                           ' —— 两边 LIVE_SCOPE_N 对不上？'
                           % (SCOPE_PATH_UI, len(raw), SCOPE_WANT))
            SCOPE["stop"].wait(0.5)
            continue
        hdr = np.frombuffer(raw[:16], dtype="<i4")
        a = np.frombuffer(raw[16:16 + 2 * SCOPE_N_UI], dtype="<i2")
        b = np.frombuffer(raw[16 + 2 * SCOPE_N_UI:], dtype="<i2")
        scope.value = (
            _panel("时域　灰=进核（麦克风）　绿=出核（送耳机）　纵轴钉死 ±满量程，已补回 96 采样群延迟",
                   _time_panel(a, b))
            + _panel("频谱　虚线=500 / 1000 / 2000 Hz 分段点　纵轴 0 ～ −100 dBFS，已补回 96 采样群延迟",
                     _fft_panel(a, b))
            + '<div style="font:12px monospace;color:#888;margin-top:4px">'
              '第 %d 帧，t=%.1f s</div>' % (hdr[1], hdr[3] / 1000.0))
        SCOPE["stop"].wait(SCOPE_SEC)


SCOPE["stop"].set()
if SCOPE["th"] is not None:
    SCOPE["th"].join(timeout=1.0)
SCOPE["stop"].clear()
SCOPE["th"] = threading.Thread(target=_scope_watch, daemon=True)
SCOPE["th"].start()

display(scope)


## 数字怎么看

| 数字 | 正常范围 | 不对时先查什么 |
|---|---|---|
| 进核 dBFS | 说话时 **−30 ～ −15** | < −50 = 麦克风没进信号。见坑 5：**要拔插到电平表跳起来为止**（实测第 35.5 秒才跳） |
| 实时倍率 | **≥ 1.00x** | 掉到 1 以下而且丢块在涨：搬样没跟上 |
| 丢块 | 每 30 秒 **个位数** | 一直涨 → 停掉重开；还涨就把 `LIVE_L` 调到 960 |
| 压缩动作 | **−4 dB 上下** | 接近 0：进核电平太低，压缩器根本没启动（它只在超过门限时才动） |

## "起跑就退了"

**同时只能有一条 `fir_live.py`。** 两条一起跑会互相踩 MMIO，而且会**同时往
同一个状态文件里写** —— 界面上的块号会来回跳，看着像"通路在抽搐"，其实是
两个进程在抢着汇报。所以 `fir_live.py` 自己用文件锁挡住了第二条。

点「启动」之后如果它两秒内就退了，脚本会把日志最后几行打出来，照那个改。
手工清场：`pkill -INT -f fir_live.py`。

## 两处"看着像坏了，其实没坏"

- **回显比滑杆慢半拍、还差零点几 dB。** 故意的：增益分 4 块（40 ms）线性走到位，
  直接跳变会在耳机里"啪"一声；差的那点小数是 Q7.8 的量化（一步 0.034 dB）。
- **拖滑杆时声音是"滑"过去的，不是瞬间换。** 同上。

## 这一页对应 PLAN §11 的哪一条

§11.2 第 ③ 条：4 条滑杆 + 预设 + 读数回显 + 电平条。
**"导入听力图"**（从一张 audiogram 自动算出每段增益）是下一步；
卡点 ①（10/21）之前做不完就砍掉它，保住滑杆。
